# Food Waste Reducer - YOLO training (Colab)
Runtime -> Change runtime type -> GPU. Ultralytics is **AGPL-3.0**. See `docs/ML.md`.

In [ ]:
!git clone <YOUR_REPO_URL> /content/repo   # or upload the ml/ folder
%cd /content/repo/ml/yolo
!pip install -q -r requirements.txt
!python -c "import ultralytics; print(ultralytics.__version__)"  # expect 8.4.175

## Fetch raw datasets
Download each source listed in `sources.example.json` into `/content/raw/<name>` (Roboflow: export as *YOLOv8* format with your API key; Kaggle: `kaggle datasets download`). Copy the JSON to `sources.json` and extend each `class_map` after reading the dataset's `data.yaml` names.

In [ ]:
# Example (set ROBOFLOW_API_KEY in Colab secrets; never paste keys into the notebook)
# !pip install -q roboflow
# from roboflow import Roboflow; from google.colab import userdata
# rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
# rf.workspace('wajeeha-workspace').project('fridge-objects-mtmmo').version(1).download('yolov8', location='/content/raw/fridge_objects')
!cp -n sources.example.json sources.json
!ls /content/raw

In [ ]:
!python prepare_dataset.py --sources sources.json --out /content/fwr_data
!cat /content/fwr_data/dataset_report.json | head -60

In [ ]:
!python train.py --data /content/fwr_data --epochs 80 --cls-epochs 30

In [ ]:
from ultralytics import YOLO
m = YOLO('runs/export/detector.pt')
print(m.val(data='/content/fwr_data/data.yaml', split='test'))  # held-out incl. synthetic low/warm copies

## Export
`runs/export/` holds `detector.pt`, `freshness.pt`, `predictions.jsonl`, `heldout_gt.jsonl`, `heldout_manifest.csv`. Hand `predictions.jsonl` + `heldout_gt.jsonl` to ML QA; put the two `.pt` files on the GPU box for `ml/serve`.

In [ ]:
from google.colab import files
!cd runs && zip -r export.zip export
files.download('runs/export.zip')